# Paper experiments: E1, E2, E3

Runs every experiment for the paper on a free Colab GPU.

| Stage | Experiment | Approx. time on a T4 |
|---|---|---|
| 0 | Setup | 3–5 min |
| 1 | Re-evaluate the original (seed 42) models and save per-product predictions | 5 min with `results.zip`, ~30 min without |
| 2 | **E1**: retrain with category words removed from names (3 seeds) | ~45 min |
| 3 | **E3**: extra seeds (7, 123) for the original models | ~55 min |
| 4 | **E2**: robustness tests | ~10 min |
| 5 | **E3**: statistics, paper tables and figures, then download | 2 min |

**Before you start**
1. *Runtime → Change runtime type → T4 GPU → Save.*
2. Upload with the 📁 panel on the left:
   - `multimodal-product-classifier-v2.zip` (the new code), and
   - `results.zip` from your last Colab run (optional but recommended: it already contains your seed-42 models and the exact same data split).
3. Run the cells **one at a time, top to bottom**.

If Colab disconnects, rerun Stage 0, then upload `paper_results_backup.zip` from your Google Drive folder `MyDrive/paper_backup/` and rerun the stage you were on. Finished runs are skipped automatically.

## Stage 0: setup

In [ ]:
import glob, os, zipfile, shutil

os.chdir("/content")
# 1) Unzip the code (works even if the browser renamed the file, or it is inside another zip)
for z in glob.glob("/content/*.zip"):
    if "results" not in os.path.basename(z) and "backup" not in os.path.basename(z):
        zipfile.ZipFile(z).extractall("/content/project")
for z in glob.glob("/content/project/**/*.zip", recursive=True):
    zipfile.ZipFile(z).extractall("/content/project")
roots = [os.path.dirname(p) for p in glob.glob("/content/project/**/scripts/paper_stats.py", recursive=True)]
assert roots, "New code not found: upload multimodal-product-classifier-v2.zip"
ROOT = os.path.dirname(roots[0])
os.chdir(ROOT)
print("Project folder:", ROOT)

# 2) Previous results (seed-42 models + data split) and/or a backup from Drive
for z in sorted(glob.glob("/content/results*.zip")) + sorted(glob.glob("/content/paper_results_backup*.zip")):
    zipfile.ZipFile(z).extractall(ROOT)
    print("Restored:", os.path.basename(z))

In [ ]:
!pip install -q -r requirements.txt hf_xet
!pip uninstall -y -q torchao
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Helpers: run a training job unless it is already finished, and back up results to Google Drive
from google.colab import drive
drive.mount("/content/drive")
BACKUP_DIR = "/content/drive/MyDrive/paper_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

def run_tag(mode, seed=42, masked=False):
    return mode + ("_masked" if masked else "") + (f"_s{seed}" if seed != 42 else "")

def done(tag):
    return os.path.exists(f"outputs/metrics/lora_{tag}.json") and os.path.exists(f"outputs/predictions/lora_{tag}.npz")

def train(mode, seed=42, masked=False):
    tag = run_tag(mode, seed, masked)
    if done(tag):
        print(f"✓ already finished: {tag}"); return
    get_ipython().system(f"python scripts/train.py --mode {mode} --seed {seed}" + (" --mask-keywords" if masked else ""))

In [ ]:
# Back up results (not the code or the dataset cache) to Google Drive after each stage
def backup():
    path = "/content/paper_results_backup.zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder in ["outputs", "data/processed"]:
            for f in glob.glob(f"{folder}/**/*", recursive=True):
                if os.path.isfile(f):
                    zf.write(f)
    shutil.copy(path, BACKUP_DIR)
    print("Backed up to Google Drive:", BACKUP_DIR)

!python scripts/prepare_data.py

## Stage 1: original models (seed 42) with saved predictions

If `results.zip` was uploaded, the checkpoints exist and are only **re-evaluated** (~1 min each), so their per-product predictions get saved. Otherwise they are trained.

In [ ]:
for mode in ["multimodal", "text", "image"]:
    if done(mode):
        print(f"✓ already finished: {mode}")
    elif os.path.exists(f"outputs/checkpoints/lora_{mode}.pt"):
        get_ipython().system(f"python scripts/evaluate.py --mode {mode}")
    else:
        train(mode)

!python scripts/train_baseline.py
backup()

## Stage 2: E1, category words removed from all names

This is the paper's **main experiment**: how well can each model classify products when the name no longer contains the answer? Image-only models are unaffected, so they are not retrained.

In [ ]:
!python scripts/train_baseline.py --mask-keywords
for seed in [42, 7, 123]:
    train("text", seed, masked=True)
    train("multimodal", seed, masked=True)
    backup()

## Stage 3: E3, extra seeds for the original models

In [ ]:
for seed in [7, 123]:
    for mode in ["text", "multimodal", "image"]:
        train(mode, seed)
    backup()

## Stage 4: E2, robustness to missing and damaged inputs

In [ ]:
!python scripts/robustness.py
backup()

## Stage 5: statistics, tables and figures

In [ ]:
!python scripts/paper_stats.py
!python scripts/compare_models.py
from IPython.display import Image, display, Markdown
display(Image("outputs/paper/shortcut_gap.png"))
display(Image("outputs/figures/robustness.png"))
display(Markdown(open("outputs/paper/summary.md").read()))

In [ ]:
backup()
!zip -qr /content/results_paper.zip outputs data/processed
from google.colab import files
files.download("/content/results_paper.zip")